# `select` — Advanced Reference

`select` picks columns by name, regex, slice, dtype, or substring — and supports `everything()` to pull in "the rest" after pinning specific columns first.

| Argument type | Example | Selects |
|---|---|---|
| List | `['a','b']` | exact named columns |
| String (regex) | `'mm$'` | columns matching regex |
| String (slice) | `'bill_length_mm:body_mass_g'` | contiguous range |
| `everything()` | after explicit cols | remaining columns |
| callable | `lambda c: c.startswith('b')` | columns passing test |
| `dtype=` | `dtype='numeric'` | by dtype |
| `exclude_dtype=` | `exclude_dtype='number'` | excluding dtype |
| `contains=` | `contains='mm'` | substring in name |
| `startswith=` | `startswith='bill'` | name prefix |
| `endswith=` | `endswith='mm'` | name suffix |

---

In [31]:
import sys, os
_src = os.path.abspath(os.path.join(os.getcwd(), '..', 'src'))
if _src not in sys.path: sys.path.insert(0, _src)

import pytae as pt

penguins = pt.sample_data['penguins']
titanic = pt.sample_data['titanic']
tips = pt.sample_data['tips']

penguins.cols(ascending=None)  # see column order

['species',
 'island',
 'bill_length_mm',
 'bill_depth_mm',
 'flipper_length_mm',
 'body_mass_g',
 'sex']

## Regex selection — multiple patterns as separate args
Passing multiple string args is cleaner than one long regex and follows OR logic.

In [32]:
# All columns that contain 'bill' OR 'mass' OR 'species'
(penguins
 .select('bill', 'mass', 'species')
 .head()
)

,bill_length_mm,bill_depth_mm,body_mass_g,species
0,39.1,18.7,3750.0,Adelie
1,39.5,17.4,3800.0,Adelie
2,40.3,18.0,3250.0,Adelie
3,NaN,NaN,NaN,Adelie
4,36.7,19.3,3450.0,Adelie


In [33]:
# Column order rule: explicit list preserves YOUR order; regex follows original DataFrame order
# island, sex, species are listed explicitly so they come first in that exact order,
# then 'mm' regex adds remaining mm-cols in original df order
(penguins
 .select(['island', 'sex', 'species'], 'mm')
 .head()
)

,island,sex,species,bill_length_mm,bill_depth_mm,flipper_length_mm
0,Torgersen,Male,Adelie,39.1,18.7,181.0
1,Torgersen,Female,Adelie,39.5,17.4,186.0
2,Torgersen,Female,Adelie,40.3,18.0,195.0
3,Torgersen,None,Adelie,NaN,NaN,NaN
4,Torgersen,Female,Adelie,36.7,19.3,193.0


In [34]:
# Columns ending in 'mm' — anchored regex
(penguins
 .select('mm$')
 .head()
)

,bill_length_mm,bill_depth_mm,flipper_length_mm
0,39.1,18.7,181.0
1,39.5,17.4,186.0
2,40.3,18.0,195.0
3,NaN,NaN,NaN
4,36.7,19.3,193.0


In [35]:
# Combining a regex arg with a slice arg in one call
# 'mm' regex picks the mm-cols; 'island:body_mass_g' slice picks everything between those two cols
(penguins
 .select('mm', 'island:body_mass_g')
 .head()
)

,bill_length_mm,bill_depth_mm,flipper_length_mm,island,body_mass_g
0,39.1,18.7,181.0,Torgersen,3750.0
1,39.5,17.4,186.0,Torgersen,3800.0
2,40.3,18.0,195.0,Torgersen,3250.0
3,NaN,NaN,NaN,Torgersen,NaN
4,36.7,19.3,193.0,Torgersen,3450.0


## Slice notation — contiguous column range
`'start_col:end_col'` selects all columns from start to end (inclusive), in DataFrame order.

In [36]:
# Select from 'bill_length_mm' through 'body_mass_g' (all measurement cols)
(penguins
 .select('bill_length_mm:body_mass_g')
 .head()
)

,bill_length_mm,bill_depth_mm,flipper_length_mm,body_mass_g
0,39.1,18.7,181.0,3750.0
1,39.5,17.4,186.0,3800.0
2,40.3,18.0,195.0,3250.0
3,NaN,NaN,NaN,NaN
4,36.7,19.3,193.0,3450.0


## `everything()` — pin key columns first, then the rest
Use `everything()` to move specific columns to the front without dropping any.

In [37]:
# Advanced regex: columns containing 'mm' but NOT 'flipper'
# this is regex for selecting all columns containing 'mm' but not 'flipper'.
# just because it can be done, does not mean it should be done.
# use more readable ways like list comprehension — this will ensure your co-workers
# don't stab you in the middle of the night
(penguins
 .select(r'^(?=.*mm)(?!.*flipper).*')  # regex sorcery
 .head()
)

# More readable alternative:
cols_with_mm_not_flipper = [c for c in penguins.columns if 'mm' in c and 'flipper' not in c]
penguins.select(cols_with_mm_not_flipper).head()

,bill_length_mm,bill_depth_mm
0,39.1,18.7
1,39.5,17.4
2,40.3,18.0
3,NaN,NaN
4,36.7,19.3


In [38]:
# Move 'body_mass_g' and 'species' to front, keep all other columns after
(penguins
 .select(['body_mass_g', 'species'], pt.everything())
 .head()
)

,body_mass_g,species,island,bill_length_mm,bill_depth_mm,flipper_length_mm,sex
0,3750.0,Adelie,Torgersen,39.1,18.7,181.0,Male
1,3800.0,Adelie,Torgersen,39.5,17.4,186.0,Female
2,3250.0,Adelie,Torgersen,40.3,18.0,195.0,Female
3,NaN,Adelie,Torgersen,NaN,NaN,NaN,None
4,3450.0,Adelie,Torgersen,36.7,19.3,193.0,Female


## Callable — select columns with a function

In [39]:
# Lambda: columns whose name has more than 6 characters
(penguins
 .select(lambda c: len(c) > 6)
 .head()
)

,species,bill_length_mm,bill_depth_mm,flipper_length_mm,body_mass_g
0,Adelie,39.1,18.7,181.0,3750.0
1,Adelie,39.5,17.4,186.0,3800.0
2,Adelie,40.3,18.0,195.0,3250.0
3,Adelie,NaN,NaN,NaN,NaN
4,Adelie,36.7,19.3,193.0,3450.0


## `dtype`, `exclude_dtype`, `contains`, `startswith`, `endswith`

In [40]:
# All numeric columns only
(penguins
 .select(dtype='numeric')
 .head()
)

,bill_length_mm,bill_depth_mm,flipper_length_mm,body_mass_g
0,39.1,18.7,181.0,3750.0
1,39.5,17.4,186.0,3800.0
2,40.3,18.0,195.0,3250.0
3,NaN,NaN,NaN,NaN
4,36.7,19.3,193.0,3450.0


In [41]:
# Drop numeric columns — keep only categorical/string columns
(penguins
 .select(exclude_dtype='number')
 .head()
)

,species,island,sex
0,Adelie,Torgersen,Male
1,Adelie,Torgersen,Female
2,Adelie,Torgersen,Female
3,Adelie,Torgersen,None
4,Adelie,Torgersen,Female


In [42]:
# exclude_dtype also accepts a list — exclude multiple dtype families at once
(penguins
 .select(exclude_dtype=['datetime', 'object'])
 .head()
)

,bill_length_mm,bill_depth_mm,flipper_length_mm,body_mass_g
0,39.1,18.7,181.0,3750.0
1,39.5,17.4,186.0,3800.0
2,40.3,18.0,195.0,3250.0
3,NaN,NaN,NaN,NaN
4,36.7,19.3,193.0,3450.0


In [43]:
# Titanic: columns starting with 'p' or 'a', plus all numeric
(titanic
 .select(startswith=['p', 'a'], dtype='numeric')
 .head()
)

,survived,pclass,age,sibsp,parch,fare,adult_male,alive,alone
0,0,3,22.0,1,0,7.2500,True,no,False
1,1,1,38.0,1,0,71.2833,False,yes,False
2,1,3,26.0,0,0,7.9250,False,yes,True
3,1,1,35.0,1,0,53.1000,False,yes,False
4,0,3,35.0,0,0,8.0500,True,no,True


In [44]:
# endswith — select columns whose name ends with a given suffix
(penguins
 .select(endswith='mm')  # same result as .select('mm$') but more readable
 .head()
)

,bill_length_mm,bill_depth_mm,flipper_length_mm
0,39.1,18.7,181.0
1,39.5,17.4,186.0
2,40.3,18.0,195.0
3,NaN,NaN,NaN
4,36.7,19.3,193.0


In [45]:
# Columns containing 'bill' or 'flipper' in name, used in a full chain
(penguins
 .qry({'species': ['Adelie', 'Gentoo']})
 .select(['species', 'island'], contains=['bill', 'flipper'])
 .agg_df(aggfunc=['mean', 'n'])
)

,species,island,n,bill_length_mm,bill_depth_mm,flipper_length_mm
0,Adelie,Biscoe,44,38.975000,18.370455,188.795455
1,Adelie,Dream,56,38.501786,18.251786,189.732143
2,Adelie,Torgersen,52,38.950980,18.429412,191.196078
3,Gentoo,Biscoe,124,47.504878,14.982114,217.186992
